# Project 19: Explainable Edge Crop Disease Detection
**Team No.:** 14  
**Team Members:** Harapriyadarshani Mishra; Manoj Kumar Behera; Jyoti Prakash Mallik; Jyotirmay Das  
**Proposed Hybrid Model:** MobileNetV3 + MobileViT  
**Dataset:** Paddy Doctor disease classification competition  
**Source:** https://www.kaggle.com/competitions/paddy-disease-classification


## 0. Setup — Environment, Imports, Reproducibility


In [ ]:
!pip -q install -U kaggle kagglehub tqdm tabulate

import os, json, random, glob, subprocess, sys, math, warnings, time, zipfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Reproducibility. Determinism can be slightly slower, but avoids silent run-to-run changes.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."

DEVICE = torch.device("cuda:0")
print("Using device:", DEVICE)
print("PyTorch:", torch.__version__)

### CONFIG


In [ ]:
CONFIG = {
    "project_no": "19",
    "project_name": "Explainable Edge Crop Disease Detection",
    "team_no": "14",
    "task_type": "classification",
    "kaggle_dataset_slug": "paddy-disease-classification",
    "target_column": None,
    "id_columns": [],
    "time_column": None,
    "split_ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
    "random_seed": SEED,
    "data_raw_dir": "data/raw",
    "data_processed_dir": "data/processed",
    "figures_dir": "figures",
    "results_dir": "results",
    "reports_dir": "reports",
    "batch_size": 32,
    "epochs": 30,
    "patience": 5,
}

for key in [
    "data_raw_dir",
    "data_processed_dir",
    "figures_dir",
    "results_dir",
    "reports_dir",
]:
    os.makedirs(CONFIG[key], exist_ok=True)

CONFIG

## 1. Download the Paddy Doctor Dataset

This notebook first checks whether the competition files are already present. If not, it tries the current `kagglehub` downloader and then the Kaggle CLI as a fallback.

**Important:** Kaggle can list competition files even when data download is blocked. If download returns `403 Forbidden` or a participation/rules error, open the Paddy Doctor competition page in Kaggle, click **Join Competition / I Understand and Accept**, then rerun this cell.

In [ ]:
from pathlib import Path
import os
import zipfile
import shutil
import subprocess


# ============================
# Configuration
# ============================

CONFIG = {
    "data_raw_dir": "/content/data/paddy_raw",
    "kaggle_json_path": "/content/kaggle.json",
    "competition_name": "paddy-disease-classification"
}


raw = Path(CONFIG["data_raw_dir"])
raw.mkdir(parents=True, exist_ok=True)


IMAGE_EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


# ============================
# Setup Kaggle Credentials
# ============================

kaggle_dir = Path("/root/.kaggle")
kaggle_dir.mkdir(
    parents=True,
    exist_ok=True
)


kaggle_json = Path(CONFIG["kaggle_json_path"])


if not kaggle_json.exists():

    raise FileNotFoundError(
        """
kaggle.json not found.

Please upload your Kaggle API file to:

/content/kaggle.json
"""
    )


# Copy kaggle.json to Kaggle config location

shutil.copy(
    kaggle_json,
    kaggle_dir / "kaggle.json"
)


os.chmod(
    kaggle_dir / "kaggle.json",
    0o600
)


print("Kaggle authentication configured successfully.")



# ============================
# Dataset Check
# ============================

def dataset_is_ready(root):

    images = any(
        p.is_file() and p.suffix.lower() in IMAGE_EXTS
        for p in root.rglob("*")
    )

    csv = any(
        p.is_file() and p.suffix.lower() == ".csv"
        for p in root.rglob("*")
    )

    return images and csv



# ============================
# Extract ZIP files
# ============================

def extract_archives(root):

    zip_files = list(root.glob("*.zip"))

    for z in zip_files:

        print("Extracting:", z.name)

        with zipfile.ZipFile(z, "r") as archive:

            archive.extractall(root)



# ============================
# Download Dataset
# ============================

if not dataset_is_ready(raw):

    print("\nDownloading Paddy Disease Classification dataset...")


    command = [
        "kaggle",
        "competitions",
        "download",
        "-c",
        CONFIG["competition_name"],
        "-p",
        str(raw)
    ]


    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )


    if result.returncode != 0:

        raise RuntimeError(
            f"""
Kaggle download failed.

{result.stderr}

Check:
1. Kaggle account is connected
2. Competition rules are accepted
3. kaggle.json is valid
"""
        )


    print(result.stdout)


    extract_archives(raw)



# ============================
# Verify Dataset
# ============================

all_files = [
    p for p in raw.rglob("*")
    if p.is_file()
]


images = [
    p for p in all_files
    if p.suffix.lower() in IMAGE_EXTS
]


csv_files = [
    p for p in all_files
    if p.suffix.lower() == ".csv"
]


print("\n==============================")
print(" Dataset Verification")
print("==============================")


print("Dataset location :", raw)
print("Total files      :", len(all_files))
print("Images           :", len(images))
print("CSV files        :", len(csv_files))


assert len(images) > 0, "Images not found"
assert len(csv_files) > 0, "CSV file not found"



print("\nCSV Files:")
for c in csv_files:
    print(c)


print("\nSample Images:")
for img in images[:5]:
    print(img)


print("\nPaddy Disease Dataset is Ready!")

## 2. Load Raw Data


In [ ]:
from PIL import Image
from torchvision import transforms, models

raw_root = Path(CONFIG["data_raw_dir"])
csvs = list(raw_root.rglob("*.csv"))

# Prefer the exact training annotation file and avoid sample_submission.csv.
train_csv = next(
    (
        p for p in csvs
        if p.stem.lower() == "train"
        or ("train" in p.stem.lower() and "sample" not in p.stem.lower())
    ),
    None,
)
assert train_csv is not None, f"Training CSV not found. CSVs discovered: {[p.name for p in csvs]}"

df = pd.read_csv(train_csv)

label_col = next(
    (c for c in df.columns if c.lower() in {"label", "class", "disease", "target"}),
    None,
)
image_col = next(
    (c for c in df.columns if c.lower() in {"image_id", "image", "filename", "file", "path"}),
    None,
)
assert label_col and image_col, f"Schema not recognized: {df.columns.tolist()}"

# Build a filename lookup. For Paddy Doctor, train.csv image_id values map to train_images.
all_images = [
    p for p in raw_root.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
]
image_lookup = {}
for p in all_images:
    # Prefer paths containing "train" when duplicate filenames exist.
    key = p.name
    current = image_lookup.get(key)
    if current is None or ("train" in str(p).lower() and "train" not in str(current).lower()):
        image_lookup[key] = p

df["path"] = df[image_col].map(
    lambda x: str(image_lookup.get(Path(str(x)).name, ""))
)
unmatched = int((df["path"] == "").sum())
if unmatched:
    warnings.warn(f"{unmatched} labeled rows had no matching image and will be removed.")

df = (
    df[df["path"] != ""]
    .drop_duplicates("path")
    .reset_index(drop=True)
)

assert len(df) > 10, "Too few labeled images were matched; check the extracted dataset structure."

target = label_col
CONFIG["target_column"] = target

print("Training CSV:", train_csv)
print("Matched dataframe shape:", df.shape)
print("\nClass distribution:")
print(df[target].value_counts())

## 3. Exploratory Data Analysis (EDA) + Data Quality Memo


In [ ]:
missing=df[[image_col,target,"path"]].isna().mean()
plt.figure(figsize=(7,3)); missing.plot.bar(); plt.tight_layout(); plt.savefig("figures/fig00_missingness.png",dpi=150); plt.show()
plt.figure(figsize=(10,4)); df[target].value_counts().plot.bar(); plt.tight_layout(); plt.savefig("figures/fig00_target_distribution.png",dpi=150); plt.show()
memo = "\n".join([
    "# Data quality memo",
    f"- Matched labeled images: {len(df)}.",
    f"- Classes: {df[target].nunique()}.",
    "- Split is stratified and path-disjoint.",
    "- Augmentation is used only for training.",
])
Path("reports/data_quality_memo.md").write_text(memo,encoding="utf-8")


## 4. Preprocessing & Feature Engineering


Feature construction is performed after splitting; every learned imputer, scaler, encoder, graph, and vocabulary is fit on training data only.


## 5. Train / Validation / Test Split


In [ ]:
if "split_ratios" not in CONFIG:
    CONFIG["split_ratios"] = {"train": 0.70, "val": 0.15, "test": 0.15}
if "random_seed" not in CONFIG:
    CONFIG["random_seed"] = SEED
if "data_processed_dir" not in CONFIG:
    CONFIG["data_processed_dir"] = "data/processed"
if "figures_dir" not in CONFIG:
    CONFIG["figures_dir"] = "figures"
if "results_dir" not in CONFIG:
    CONFIG["results_dir"] = "results"
if "reports_dir" not in CONFIG:
    CONFIG["reports_dir"] = "reports"
if "batch_size" not in CONFIG:
    CONFIG["batch_size"] = 32
if "epochs" not in CONFIG:
    CONFIG["epochs"] = 30
if "patience" not in CONFIG:
    CONFIG["patience"] = 5

for key in [
    "data_processed_dir",
    "figures_dir",
    "results_dir",
    "reports_dir",
]:
    os.makedirs(CONFIG[key], exist_ok=True)

train_ratio = CONFIG["split_ratios"]["train"]
val_ratio = CONFIG["split_ratios"]["val"]
test_ratio = CONFIG["split_ratios"]["test"]
assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-9

train_df, rest = train_test_split(
    df,
    test_size=(1.0 - train_ratio),
    stratify=df[target],
    random_state=SEED,
)

relative_test = test_ratio / (val_ratio + test_ratio)
val_df, test_df = train_test_split(
    rest,
    test_size=relative_test,
    stratify=rest[target],
    random_state=SEED,
)

assert set(train_df.path).isdisjoint(val_df.path)
assert set(train_df.path).isdisjoint(test_df.path)
assert set(val_df.path).isdisjoint(test_df.path)

le = LabelEncoder().fit(train_df[target])
assert set(le.classes_) == set(df[target].unique()), "A class is missing from the training split."

IMG = 224

train_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=.15, contrast=.15, saturation=.15),
    transforms.ToTensor(),
    transforms.Normalize([.485, .456, .406], [.229, .224, .225]),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize([.485, .456, .406], [.229, .224, .225]),
])

manifest = {
    "train": len(train_df),
    "val": len(val_df),
    "test": len(test_df),
    "classes": le.classes_.tolist(),
    "seed": SEED,
}
Path(CONFIG["data_processed_dir"], "split_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)
print(manifest)


## 6. PyTorch Dataset & DataLoader


In [ ]:
class CropDataset(Dataset):
    def __init__(self, frame, tf):
        self.df = frame.reset_index(drop=True)
        self.tf = tf

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        with Image.open(row["path"]) as im:
            x = self.tf(im.convert("RGB"))
        y = torch.tensor(le.transform([row[target]])[0], dtype=torch.long)
        return x, y

loader_kwargs = {
    "batch_size": CONFIG["batch_size"],
    "num_workers": 2,
    "pin_memory": torch.cuda.is_available(),
}

train_loader = DataLoader(
    CropDataset(train_df, train_tf),
    shuffle=True,
    **loader_kwargs,
)
val_loader = DataLoader(
    CropDataset(val_df, eval_tf),
    shuffle=False,
    **loader_kwargs,
)
test_loader = DataLoader(
    CropDataset(test_df, eval_tf),
    shuffle=False,
    **loader_kwargs,
)

xb_check, yb_check = next(iter(train_loader))
print("Batch:", xb_check.shape, yb_check.shape)

## 7. Proposed Model Definition


In [ ]:
class MobileViTBlock(nn.Module):
    """Lightweight local-global block applied to the final MobileNetV3 feature map."""

    def __init__(self, c):
        super().__init__()
        self.local = nn.Sequential(nn.Conv2d(c, c, 3, padding=1, groups=c, bias=False), nn.BatchNorm2d(c), nn.Hardswish())
        encoder = nn.TransformerEncoderLayer(d_model=c, nhead=4, dim_feedforward=2 * c, dropout=0.1, batch_first=True)
        self.global_attn = nn.TransformerEncoder(encoder, num_layers=1)
        self.fuse = nn.Sequential(nn.Conv2d(2 * c, c, 1, bias=False), nn.BatchNorm2d(c), nn.Hardswish())

    def forward(self, x):
        local = self.local(x)
        b, c, h, w = local.shape
        tokens = local.flatten(2).transpose(1, 2)
        global_tokens = self.global_attn(tokens)
        global_map = global_tokens.transpose(1, 2).reshape(b, c, h, w)
        return self.fuse(torch.cat([local, global_map], dim=1)) + x

class MobileNetMobileViT(nn.Module):

    def __init__(self, k):
        super().__init__()
        base = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        self.features = base.features
        feature_channels = base.classifier[0].in_features
        self.mobilevit = MobileViTBlock(feature_channels)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Dropout(p=0.2), nn.Linear(feature_channels, k))

    def forward(self, x):
        x = self.features(x)
        x = self.mobilevit(x)
        x = self.pool(x).flatten(1)
        return self.head(x)


## 8. Training Loop


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        if is_training:
            optimizer.zero_grad(set_to_none=True)
        out = model(xb)
        loss = criterion(out, yb)
        if is_training:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * len(yb)
        n += len(yb)
    return total_loss / max(n, 1)

def _safe_load_state_dict(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=device)

def train_model(model, train_loader, val_loader, checkpoint):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=0.0003, weight_decay=0.0001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5)
    history = {'train_loss': [], 'val_loss': []}
    best = float('inf')
    stale = 0
    for epoch in tqdm(range(CONFIG['epochs']), desc='Training', unit='epoch'):
        tr = run_epoch(model, train_loader, criterion, opt)
        with torch.no_grad():
            va = run_epoch(model, val_loader, criterion)
        history['train_loss'].append(tr)
        history['val_loss'].append(va)
        scheduler.step(va)
        print(f"epoch={epoch + 1:02d} train={tr:.5f} val={va:.5f} lr={opt.param_groups[0]['lr']:.2e}")
        if va < best:
            best = va
            stale = 0
            torch.save(model.state_dict(), checkpoint)
        else:
            stale += 1
            if stale >= CONFIG['patience']:
                print(f'Early stopping at epoch {epoch + 1}.')
                break
    model.load_state_dict(_safe_load_state_dict(checkpoint, DEVICE))
    return (model, history)

def predict(model, loader):
    model.eval()
    probs_all = []
    true_all = []
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb.to(DEVICE, non_blocking=True)).cpu()
            probs_all.append(torch.softmax(out, dim=1))
            true_all.append(yb.cpu())
    return (torch.cat(probs_all).numpy(), torch.cat(true_all).numpy())
hybrid = MobileNetMobileViT(len(le.classes_))
hybrid, hybrid_history = train_model(hybrid, train_loader, val_loader, str(Path(CONFIG['results_dir']) / 'best_hybrid.pt'))


## 9. Evaluation Metrics


In [ ]:
def model_size_mb(model):
    tmp = Path(CONFIG['results_dir']) / '_tmp_model_state.pt'
    torch.save(model.state_dict(), tmp)
    size = tmp.stat().st_size / 1024 ** 2
    tmp.unlink(missing_ok=True)
    return size

def average_latency_ms(model, image_size=224, warmup=10, runs=30):
    model.eval()
    x = torch.randn(1, 3, image_size, image_size, device=DEVICE)
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(x)
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(runs):
            _ = model(x)
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
    return 1000.0 * elapsed / runs
results = {}
cached = {}
for name, model in [('hybrid', hybrid)]:
    probs, y = predict(model, test_loader)
    pred = probs.argmax(axis=1)
    cached[name] = (probs, pred, y)
    pr, re, f1, _ = precision_recall_fscore_support(y, pred, average='macro', zero_division=0)
    params = sum((p.numel() for p in model.parameters()))
    results[name] = {'accuracy': float(accuracy_score(y, pred)), 'precision_macro': float(pr), 'recall_macro': float(re), 'f1_macro': float(f1), 'parameters_million': float(params / 1000000.0), 'model_size_mb': float(model_size_mb(model)), 'latency_ms_per_image_batch1': float(average_latency_ms(model, IMG))}
Path(CONFIG['results_dir'], 'metrics.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
results


## 10. Required Figures


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(hybrid_history['train_loss'], label='hybrid train')
plt.plot(hybrid_history['val_loss'], label='hybrid val')
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.legend()
plt.tight_layout()
plt.savefig(Path(CONFIG['figures_dir']) / 'fig01_loss_curves.png', dpi=150)
plt.show()
probs, pred, y = cached['hybrid']
cm = confusion_matrix(y, pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(Path(CONFIG['figures_dir']) / 'fig02_confusion_matrix.png', dpi=150)
plt.show()
per_class = [(pred[y == i] == i).mean() if np.any(y == i) else np.nan for i in range(probs.shape[1])]
plt.figure(figsize=(10, 4))
plt.bar(le.classes_, per_class)
plt.ylabel('Recall')
plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(Path(CONFIG['figures_dir']) / 'fig03_class_performance.png', dpi=150)
plt.show()
feature_maps = {}
gradients = {}
target_layer = hybrid.features[-1]

def _save_feature_map(module, inputs, output):
    feature_maps['value'] = output

def _save_gradient(module, grad_input, grad_output):
    gradients['value'] = grad_output[0]
forward_handle = target_layer.register_forward_hook(_save_feature_map)
backward_handle = target_layer.register_full_backward_hook(_save_gradient)
try:
    xb, yb = next(iter(test_loader))
    xb = xb[:min(8, len(xb))].to(DEVICE)
    hybrid.zero_grad(set_to_none=True)
    logits = hybrid(xb)
    chosen = logits.argmax(dim=1)
    logits.gather(1, chosen[:, None]).sum().backward()
    assert 'value' in feature_maps and 'value' in gradients, 'Grad-CAM hooks did not capture tensors.'
    weights = gradients['value'].mean((2, 3), keepdim=True)
    cams = torch.relu((weights * feature_maps['value']).sum(1, keepdim=True))
    cams = torch.nn.functional.interpolate(cams, size=xb.shape[-2:], mode='bilinear', align_corners=False)
    cams = cams / (cams.amax((2, 3), keepdim=True) + 1e-08)
finally:
    forward_handle.remove()
    backward_handle.remove()
mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE)[None, :, None, None]
std = torch.tensor([0.229, 0.224, 0.225], device=DEVICE)[None, :, None, None]
images = (xb * std + mean).clamp(0, 1).detach().cpu()
cams = cams.detach().cpu()
fig, axes = plt.subplots(2, len(images), figsize=(3 * len(images), 6), squeeze=False)
for i in range(len(images)):
    image_np = images[i].permute(1, 2, 0)
    axes[0, i].imshow(image_np)
    axes[0, i].axis('off')
    axes[0, i].set_title(f'Pred: {le.classes_[chosen[i].item()]}')
    axes[1, i].imshow(image_np)
    axes[1, i].imshow(cams[i, 0], cmap='jet', alpha=0.45)
    axes[1, i].axis('off')
    axes[1, i].set_title('Grad-CAM')
plt.tight_layout()
plt.savefig(Path(CONFIG['figures_dir']) / 'fig04_gradcam.png', dpi=150)
plt.show()
errors = pred != y
plt.figure(figsize=(8, 4))
if np.any(errors):
    plt.hist(probs.max(axis=1)[errors], bins=20)
    plt.xlabel('Maximum predicted probability')
    plt.ylabel('Misclassified images')
else:
    plt.text(0.5, 0.5, 'No misclassified test images', ha='center', va='center', transform=plt.gca().transAxes)
plt.title('Confidence of misclassified images')
plt.tight_layout()
plt.savefig(Path(CONFIG['figures_dir']) / 'fig05_error_analysis.png', dpi=150)
plt.show()
metric = 'f1_macro'
plt.figure(figsize=(6, 4))
plt.bar(list(results.keys()), [results[k][metric] for k in results])
plt.ylabel(metric)
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(Path(CONFIG['figures_dir']) / 'fig06_proposed_metrics.png', dpi=150)
plt.show()
